# Unidad 1: Profundización en Programación Estructurada y Algoritmos
**Materia:** Programación — Licenciatura en Negocios Digitales (3º año)
**Institución:** Universidad del Museo Social Argentino (UMSA)

---

## Introducción y Contexto de Negocio

En el ámbito de los **Negocios Digitales**, el desarrollo de software no se limita a escribir scripts aislados para analizar datos. Diseñar productos digitales escalables, integrar pasarelas de pago o estructurar pipelines de automatización exige un dominio profundo de la lógica estructurada, el control de flujo y la modularización en Python.

Un código profesional debe ser legible, modular, tolerante a fallos y alineado con los estándares de la industria (PEP 8). En esta unidad consolidaremos las bases algorítmicas, avanzaremos en el diseño de funciones y aprenderemos a gestionar excepciones de forma estructurada para prevenir caídas de producción en nuestras aplicaciones.

### Objetivos de Aprendizaje:
1. Consolidar el uso de sintaxis fundamental de Python (loops, condicionales y colecciones).
2. Manejar tipos de datos avanzados y estructuras de datos dinámicas.
3. Controlar errores mediante bloques `try-except-else-finally` y diseñar excepciones personalizadas.
4. Crear funciones modulares, reutilizables y tipadas con anotación de tipos (type hinting).
5. Aplicar buenas prácticas de código siguiendo los lineamientos de PEP 8.


## 1. Repaso y Consolidación de Sintaxis Fundamental

Antes de pasar a arquitecturas más complejas, repasemos herramientas fundamentales de control de flujo y colecciones, enfocadas en la optimización del código (como list y dict comprehensions).


In [ ]:
# Ejemplo de List Comprehension para procesar una lista de montos de transacciones en bruto
montos_sucios = [" $120.50 ", " $45.00", "$9.99 ", "  $1500.00  "]

# Limpiamos y convertimos a float en una sola línea
montos_limpios = [float(monto.strip().replace("$", "")) for monto in montos_sucios]
print("Montos limpios:", montos_limpios)

# Filtrar transacciones de alto valor (mayores a 50)
transacciones_vip = [monto for monto in montos_limpios if monto > 50]
print("Transacciones VIP (> 50):", transacciones_vip)


### Estructuras de Datos Avanzadas: Diccionarios y Colecciones

Los diccionarios son el estándar de facto para representar payloads de APIs y configuraciones. Python cuenta con el módulo `collections` que provee estructuras de datos muy útiles para el día a día en un negocio digital (como `defaultdict` y `Counter`).


In [ ]:
from collections import defaultdict, Counter

# Imaginemos un log de eventos de navegación en nuestro e-commerce
eventos_web = [
    ("user_1", "page_view"),
    ("user_2", "add_to_cart"),
    ("user_1", "add_to_cart"),
    ("user_3", "page_view"),
    ("user_1", "checkout_click"),
    ("user_2", "checkout_click")
]

# Agrupar eventos por usuario usando defaultdict
eventos_por_usuario = defaultdict(list)
for usuario, evento in eventos_web:
    eventos_por_usuario[usuario].append(evento)

print("Eventos por usuario:")
for usuario, eventos in eventos_por_usuario.items():
    print(f" - {usuario}: {eventos}")

# Contar frecuencias de eventos en todo el sitio
conteo_eventos = Counter([evento for _, evento in eventos_web])
print("\nFrecuencia total de eventos:", dict(conteo_eventos))


## 2. Gestión Estructurada de Excepciones

En producción, **los errores van a ocurrir**: la conexión de red con una pasarela de pago puede caerse, una base de datos puede estar saturada o un cliente puede mandar un campo vacío. Si no controlamos esto, nuestra app se interrumpirá.

### Bloque `try-except-else-finally`


In [ ]:
def procesar_descuento(precio_base, descuento):
    try:
        # Intentamos calcular el precio con descuento
        precio_final = precio_base - (precio_base * (descuento / 100))
    except ZeroDivisionError:
        print("Error: El descuento no puede dividirse por cero de esta forma.")
        precio_final = precio_base
    except TypeError as e:
        print(f"Error de tipos detectado: {e}")
        precio_final = None
    else:
        print("El descuento se calculó exitosamente.")
    finally:
        print("Operación de cálculo completada.")
    return precio_final

print("--- Caso Exitoso ---")
print("Total:", procesar_descuento(100.0, 15))

print("\n--- Caso Fallido (Tipo Incorrecto) ---")
print("Total:", procesar_descuento(100.0, "quince"))


### Excepciones Personalizadas para Modelos de Negocio

Para hacer que nuestro código sea más descriptivo, es una buena práctica heredar de la clase base `Exception` para crear errores de dominio propios de nuestro negocio.


In [ ]:
# Definición de excepciones de negocio
class LimiteCreditoSuperadoError(Exception):
    def __init__(self, monto_compra, limite_disponible):
        self.monto_compra = monto_compra
        self.limite_disponible = limite_disponible
        super().__init__(f"No se pudo procesar la compra por ${monto_compra:.2f}. Límite disponible: ${limite_disponible:.2f}")

def checkout(monto, limite_usuario):
    if monto > limite_usuario:
        raise LimiteCreditoSuperadoError(monto, limite_usuario)
    return "Pago aprobado de forma exitosa!"

# Simulando el flujo de negocio
limite_tarjeta = 500.0
compras = [120.0, 450.0]

for compra in compras:
    try:
        print(f"Intentando compra por ${compra}...")
        resultado = checkout(compra, limite_tarjeta)
        print(resultado)
    except LimiteCreditoSuperadoError as e:
        print(f"ALERTA BACKEND: {e}")
        # Aquí enviaríamos una alerta o pediríamos otro método de pago


## 3. Creación de Funciones Modulares y Manejo de Módulos Locales

Escribir funciones modulares nos permite dividir problemas complejos en partes sencillas y reutilizar el código. En Python, es altamente recomendado utilizar **Type Hinting** (anotación de tipos) para mejorar la autocompletación y detectar errores antes de ejecutar el código.

### Modularización, Parámetros Dinámicos (`*args` y `**kwargs`) e Inmutabilidad


In [ ]:
from typing import List, Dict

# Función que acepta argumentos posicionales variables (*args) y palabras clave (**kwargs)
def registrar_pedido(cliente_id: int, *items: str, **detalles_envio: str) -> Dict:
    """
    Registra un pedido de forma modular con tipado estricto.
    """
    pedido = {
        "cliente_id": cliente_id,
        "productos": list(items),
        "envio": detalles_envio,
        "estado": "pendiente"
    }
    return pedido

# Creación de pedido con ítems variables y metadata de envío dinámica
pedido_ejemplo = registrar_pedido(
    1024,
    "Suscripción SaaS Pro", "Soporte Premium 24/7",
    direccion="Av. Corrientes 1500, CABA",
    metodo="Envío Express Digital",
    prioridad="Alta"
)
print("Pedido Registrado:")
print(json.dumps(pedido_ejemplo, indent=2, ensure_ascii=False))


### Manejo de Módulos Locales

En proyectos reales de desarrollo, dividimos el código en diferentes archivos `.py` (módulos) y carpetas (paquetes). 
En Google Colab o Jupyter, podemos simular la escritura de archivos locales mediante el comando mágico `%%writefile`.


In [ ]:
%%writefile validador_negocio.py
# Este código se guardará en un archivo local llamado validador_negocio.py
import re

def es_email_valido(email: str) -> bool:
    """Valida si un string tiene formato de email corporativo o general."""
    patron = r'^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$'
    return bool(re.match(patron, email))

def es_cuit_valido(cuit: str) -> bool:
    """Valida formato básico de CUIT (11 dígitos sin guiones)."""
    return cuit.isdigit() and len(cuit) == 11


Ahora podemos importar las funciones de nuestro módulo recién creado tal como lo haríamos en un proyecto profesional local.


In [ ]:
# Importamos el módulo local creado dinámicamente
from validador_negocio import es_email_valido, es_cuit_valido

emails = ["juan@empresa.com", "cliente_invalido.com"]
cuits = ["20351234567", "123-45"]

for em in emails:
    print(f"¿Email '{em}' es válido?:", es_email_valido(em))

for cu in cuits:
    print(f"¿CUIT '{cu}' es válido?:", es_cuit_valido(cu))


## 4. Buenas Prácticas de Código y Estándares (PEP 8)

El estándar **PEP 8** define las reglas de estilo de Python:
- Nombre de variables en minúscula separadas por guiones bajos (`mi_variable`).
- Nombre de clases en CamelCase (`ConfiguracionSaaS`).
- Sangrado (indentación) de 4 espacios (evitar tabuladores).
- Comentarios explicativos concisos y docstrings descriptivos para todas las funciones.

Herramientas automáticas recomendadas para verificar la calidad en proyectos locales:
1. **Black**: Formateador automático que reestructura tu código según los estándares más estrictos.
2. **Ruff / Flake8**: Linters que analizan estáticamente tu código en busca de bugs latentes, variables no usadas y violaciones de estilo.

---

## Desafío Práctico (Trabajo Práctico 1)

**Consigna de Negocio (Pipeline de Clientes):**
Tu startup de Negocios Digitales necesita procesar un lote de nuevos clientes (leads) en bruto. Debes implementar un pipeline modular.

1. Crea un módulo local llamado `procesador_leads.py` utilizando la celda mágica `%%writefile`.
2. Dentro del módulo, define una excepción personalizada llamada `LeadInvalidoError` que contenga el email del lead y la razón de la falla.
3. Escribe una función `limpiar_y_validar_lead(lead: dict) -> dict` que reciba un diccionario del lead (con campos `nombre`, `email`, y `cuit`) y devuelva un diccionario limpio (con espacios eliminados y campos validados). 
   - Debe lanzar `LeadInvalidoError` si el email no es válido (usa la función del módulo `validador_negocio` creado antes) o si el cuit no es numérico de 11 dígitos.
4. Escribe una función principal en el cuaderno que itere sobre la lista de leads provista abajo, procese cada uno capturando excepciones de forma que un lead inválido **no interrumpa** el procesamiento de los demás, e imprima un reporte final de leads exitosos y leads fallidos.

A continuación, implementa tu solución y pruébala.


In [ ]:
# 1. Escribe el código del archivo procesador_leads.py usando la celda mágica
# %%writefile procesador_leads.py
# ...


In [ ]:
# 2. Importa tus funciones y ejecuta el flujo de prueba
leads_en_bruto = [
    {"nombre": "  Ana Gómez  ", "email": "ana@empresa.com", "cuit": "27958432168"},
    {"nombre": "Carlos Pérez", "email": "carlos-fallido.com", "cuit": "20325418967"},
    {"nombre": "  Sofía Ruiz ", "email": "sofia@startup.co", "cuit": "20448123C45"},
    {"nombre": "Juan López", "email": "juan@corporativo.info", "cuit": "20112233445"}
]

# --- Implementa tu bucle de procesamiento aquí ---
# ...
